# RA2CE Notebook to get graphs and perform  hazard overlay

Do your imports

In [1]:
# === Standard Library ===
from pathlib import Path
import pickle

# === Scientific & Data Libraries ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm


# === Geospatial Libraries ===
import geopandas as gpd
import rasterio
import folium
from shapely.geometry import box, Point, LineString, Polygon, shape
from pyproj import Transformer
import networkx as nx
import geohexgrid as ghg
from shapely.ops import transform


# === RA2CE Project Imports ===
from ra2ce.network.network_config_data.enums.aggregate_wl_enum import AggregateWlEnum
from ra2ce.network.network_config_data.enums.source_enum import SourceEnum
from ra2ce.network.network_config_data.enums.network_type_enum import NetworkTypeEnum
from ra2ce.network.network_config_data.enums.road_type_enum import RoadTypeEnum
from ra2ce.network.network_config_data.network_config_data import (
    HazardSection,
    NetworkConfigData,
    NetworkSection,
    OriginsDestinationsSection
)
from ra2ce.network.exporters.geodataframe_network_exporter import GeoDataFrameNetworkExporter
from ra2ce.network.exporters.multi_graph_network_exporter import MultiGraphNetworkExporter
from ra2ce.network.network_wrappers.osm_network_wrapper.osm_network_wrapper import OsmNetworkWrapper
from ra2ce.ra2ce_handler import Ra2ceHandler

import os
from pathlib import Path


c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
# specify the name of the path to the project folder where you created the RA2CE folder setup

root_dir = Path(r'N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse')
#root_dir = Path.cwd().parent.joinpath("data")
assert root_dir.exists()
static_path = root_dir.joinpath("static_maxwd")
network_path = static_path.joinpath("network")
output_path = static_path.joinpath("output_graph")
hazard_path = static_path.joinpath("hazard", "reprojected")


## Find the study area

In [6]:
# some preliminary functions

def get_all_files(directory: str) -> list[Path]:
    p = Path(directory)
    return [file for file in p.iterdir() if file.is_file()]

def read_pickle(file_path: str):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return data

def read_gpkg_to_gdf(file_path: str, layer: str = None) -> gpd.GeoDataFrame:
    # Read the geopackage file into a GeoDataFrame
    gdf = gpd.read_file(file_path, layer=layer)
    return gdf

In [33]:
hazard_files = get_all_files(hazard_path)
hazard_crs = "EPSG:4326" 

for hazard_file in hazard_files:
    print (hazard_file)

N:\Projects\11209000\11209175\B. Measurements and calculations\Data\full_run\V_K_analyse\static_maxwd\hazard\reprojected\Merge_maxWD_ZH_200mm_Nat_processed_wgs84.tif


In [35]:
import rasterio
from shapely.geometry import box

# Path to your GeoTIFF file
geotiff_path = hazard_file

# Open the GeoTIFF and get its bounds
with rasterio.open(geotiff_path) as src:
    bounds = src.bounds
    crs = src.crs  # Coordinate Reference System

# Create a Shapely polygon from the bounds
bbox_polygon = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
gdf_polygon = gpd.GeoDataFrame({'geometry': [bbox_polygon]}, crs=crs)

# Print the polygon and CRS
print("Bounding Box Polygon:", bbox_polygon)
print("CRS:", crs)


Bounding Box Polygon: POLYGON ((4.885864898301681 51.648965206432415, 4.885864898301681 52.45857655486167, 3.814146231690125 52.45857655486167, 3.814146231690125 51.648965206432415, 4.885864898301681 51.648965206432415))
CRS: EPSG:4326


In [39]:
gdf_polygon.to_file(network_path.joinpath("polygon.geojson"), driver='GeoJSON')

# RA2CE

### Let us first initalize and perform the ra2ce run so we have all the data that we need

#### Cutting RoadTypeEnum.MOTORWAY,RoadTypeEnum.MOTORWAY_LINK to make analysis more realistic

In [36]:
gdf_polygon_path = network_path.joinpath("polygon.geojson")


In [40]:
_network_section = NetworkSection(
    network_type=NetworkTypeEnum.DRIVE,
    source=SourceEnum.OSM_DOWNLOAD,
    polygon=gdf_polygon_path, #it needs a path without the list!
    save_gpkg=True,
    road_types=[RoadTypeEnum.MOTORWAY, RoadTypeEnum.MOTORWAY_LINK, RoadTypeEnum.PRIMARY, RoadTypeEnum.PRIMARY_LINK,RoadTypeEnum.TRUNK, RoadTypeEnum.SECONDARY,RoadTypeEnum.SECONDARY_LINK, RoadTypeEnum.TERTIARY, RoadTypeEnum.RESIDENTIAL, RoadTypeEnum.UNCLASSIFIED], 
    attributes_to_exclude_in_simplification=['bridge', 'tunnel'],
)

# Make the NetworkConfigData
_hazard_section = HazardSection(
    hazard_map=hazard_files,
    hazard_id=None,
    hazard_field_name="waterdepth",
    aggregate_wl=AggregateWlEnum.MAX,
    hazard_crs=hazard_crs,
    overlay_segmented_network = True
)

_network_config_data = NetworkConfigData(
    root_path=root_dir,
    static_path=static_path,
    output_path=output_path,
    network=_network_section,
    hazard=_hazard_section)


In [ ]:
# Run analysis
_handler = Ra2ceHandler.from_config(_network_config_data, analysis=None)
_handler.configure()


c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\osmnx\simplification.py:513: UserWarning: Geometry is in a geographic CRS. Results from 'buffer' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  merged = convert.graph_to_gdfs(G, edges=False)["geometry"].buffer(tolerance).unary_union
c:\Users\meije_le\AppData\Local\miniforge3\envs\ra2ce_env\Lib\site-packages\osmnx\simplification.py:560: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = node_clusters.centroid


In [ ]:
# # Create base_graph_hazard_editted
# from grid_based_accessibility_hex import read_pickle
# import pickle 

# input_graph = Path(r'C:\repos\powerpath\data\static\output_graph\base_graph_hazard.p')
# output_graph = Path(r'C:\repos\powerpath\data\static\output_graph\base_graph_hazard_editted.p')
# if not output_graph.exists():
#     base_graph = read_pickle(input_graph)
#     base_graph_hazard_editted = base_graph.copy()

# # Add column EV0_ma and set to 0.0, also set EV6-9 to 0.0
# for i in range(10):
#     col_name = f'EV{i}_ma'
    
#     # For NetworkX graphs, we need to add attributes to each edge
#     for u, v, key in base_graph_hazard_editted.edges(keys=True):
#         # Check if the attribute exists on any edge, if not add it
#         if col_name not in base_graph_hazard_editted[u][v][key]:
#             base_graph_hazard_editted[u][v][key][col_name] = 0.0
        
#         # Set EV6-9 to 0.0
#         if i >= 6:
#             base_graph_hazard_editted[u][v][key][col_name] = 0.0

# for u,v,d in base_graph_hazard_editted.edges(data=True):
#     print (d['EV0_ma'], d['EV1_ma'], d['EV2_ma'], d['EV3_ma'], d['EV4_ma'], d['EV5_ma'], d['EV6_ma'], d['EV7_ma'], d['EV8_ma'], d['EV9_ma'])

# pickle.dump(base_graph_hazard_editted, open(output_graph, 'wb'))  # Save the modified graph
#     # Save the modified graph
